# M07 — Parquet y partición de negocio

[← Anterior](../M06-optimizacion-ejecucion/03-lab-cache-particionado.ipynb) · [Siguiente →](02-lab-parquet-layout.ipynb)

El pipeline no acaba en un `show()`. Acaba en un **directorio** que otro proceso (o tú mañana) puede leer sin repetir joins.

Dos “particiones” que se confunden:

- `repartition` (M06) baraja **memoria** entre tareas.
- `partitionBy` en el `write` crea **carpetas en disco** (`order_month=2024-01`, …). Al filtrar un mes, Spark puede **no abrir** las demás.

Ejecuta las celdas **aquí**, en este mismo fichero (clase, juntos). Va **montado**: explicación + código + lo que tienes que ver. Lo que construyes tú está en el **lab**.

Kernel: **Python (NovaShop)**.


## Arranque

La primera celda **no es Spark todavía**: busca la raíz del repo (aunque este notebook no esté en la carpeta de arriba) y deja `RAW`, `STAGING` y `CURATED` listos. La segunda pide una `SparkSession` en `local[*]` (todos los cores de esta máquina; no hay clúster).

Al ejecutar: rutas impresas y una versión `3.5.x` con master `local[*]`.


In [ ]:
import sys
from pathlib import Path

# El notebook puede estar en trabajo/; subimos hasta encontrar el repo.
_here = Path.cwd().resolve()
ROOT = next(
    p
    for p in [_here, *_here.parents]
    if (p / "labs" / "_shared" / "session.py").is_file()
)
sys.path.insert(0, str(ROOT / "labs" / "_shared"))

from paths import RAW, STAGING, CURATED  # rutas absolutas, no Path("data/raw")
from session import get_spark

print("ROOT   ", ROOT)
print("RAW    ", RAW, "existe:", RAW.is_dir())
print("STAGING", STAGING)
print("CURATED", CURATED)


In [ ]:
# getOrCreate: si ya hay sesión en este kernel, la reusa (mismo puerto 4040)
spark = get_spark('novashop-clase-m07')
print(spark.version, spark.sparkContext.master)


## Escribes carpetas, no un Excel

Tres filas de juguete, dos meses. `write.partitionBy("order_month").parquet(...)` deja un directorio por mes. `overwrite` hace el resultado **idempotente**: si vuelves a ejecutar, no duplicas.

Al leer solo enero, `explain` debería mencionar `2024-01` (o un *PartitionFilters*). El count de enero es **2**; el total, **3**.


In [ ]:
from pyspark.sql import Row
from pyspark.sql.functions import col

demo = spark.createDataFrame([
    Row(order_id="O1", order_month="2024-01", gmv=10.0),
    Row(order_id="O2", order_month="2024-01", gmv=20.0),
    Row(order_id="O3", order_month="2024-02", gmv=5.0),
])
dest = CURATED / "_demo_sales"
CURATED.mkdir(parents=True, exist_ok=True)
# Carpetas en disco, no un único fichero "datos.parquet"
demo.write.mode("overwrite").partitionBy("order_month").parquet(str(dest))
print("carpetas:", sorted(p.name for p in dest.iterdir() if p.is_dir()))

enero = spark.read.parquet(str(dest)).where(col("order_month") == "2024-01")
enero.explain("formatted")  # busca 2024-01 / PartitionFilters
print("enero", enero.count(), "total", spark.read.parquet(str(dest)).count())


**Siguiente:** [lab de parquet](02-lab-parquet-layout.ipynb) sobre el fact real.
